In [5]:
class IndexService:

    def __init__(self):
        self.documents = {}
        self.index = {}

    def add_document(self, doc_data):

        doc_id = str(len(self.documents) + 1)

        self.documents[doc_id] = {
            **doc_data,
            "id": doc_id
        }

        words = doc_data["content"].lower().split()

        for word in words:

            if word not in self.index:
                self.index[word] = set()

            self.index[word].add(doc_id)

        return self.documents[doc_id]

    def get_document(self, doc_id):
        return self.documents.get(doc_id)

    def search_word(self, word):
        return list(
            self.index.get(word.lower(), set())
        )


class QueryService:

    def __init__(self, index_service):

        self.index_service = index_service
        self.queries = {}

    def create_query(self, query_data):

        query_id = str(len(self.queries) + 1)

        search_terms = query_data["terms"]

        operator = query_data.get(
            "operator",
            "AND"
        ).upper()

        #AND SEARCH

        if operator == "AND":

            results = None

            for term in search_terms:

                doc_ids = set(
                    self.index_service.search_word(term)
                )

                if results is None:
                    results = doc_ids

                else:
                    results = results & doc_ids

            if results is None:
                results = set()

        #OR SEARCH

        elif operator == "OR":

            results = set()

            for term in search_terms:

                doc_ids = set(
                    self.index_service.search_word(term)
                )

                results = results | doc_ids

        else:

            return {
                "error": "Invalid operator"
            }

        #RANKING

        ranked_results = []

        for doc_id in results:

            doc = self.index_service.get_document(doc_id)

            score = 0

            for term in search_terms:

                score += doc["content"].lower().count(
                    term.lower()
                )

            ranked_results.append({
                "doc_id": doc_id,
                "score": score
            })

        ranked_results.sort(
            key=lambda x: x["score"],
            reverse=True
        )

        query = {
            "id": query_id,
            "terms": search_terms,
            "operator": operator,
            "results": ranked_results
        }

        self.queries[query_id] = query

        return query


class ResultService:

    def __init__(
        self,
        index_service,
        query_service
    ):

        self.index_service = index_service
        self.query_service = query_service
        self.results = {}

    def format_results(self, query_id):

        query = self.query_service.queries.get(query_id)

        if not query:

            return {
                "error": "Query not found"
            }

        formatted_results = []

        for item in query["results"]:

            doc_id = item["doc_id"]

            score = item["score"]

            doc = self.index_service.get_document(doc_id)

            if doc:

                formatted_results.append({

                    "doc_id": doc_id,

                    "title": doc["title"],

                    "snippet":
                    doc["content"][:100] + "...",

                    "score": score
                })

        result_id = str(len(self.results) + 1)

        result = {

            "id": result_id,

            "query_id": query_id,

            "formatted_results": formatted_results,

            "count": len(formatted_results)
        }

        self.results[result_id] = result

        return result


def main():

    index_service = IndexService()

    query_service = QueryService(
        index_service
    )

    result_service = ResultService(
        index_service,
        query_service
    )

    #ADD DOCUMENTS

    #Document1
    index_service.add_document({

        "title": "Python Programming",

        "content":
        "Python is a popular programming language for cloud computing"
    })

    #Document2
    index_service.add_document({

        "title": "Cloud Services",

        "content":
        "Cloud computing enables scalable microservices architecture"
    })

    #Document3
    #We added this document
    index_service.add_document({

        "title": "Microservices",

        "content":
        "Microservices architecture improves scalability"
    })

    #AND QUERY

    query1 = query_service.create_query({

        "terms": ["cloud", "computing"],

        "operator": "AND"
    })

    print("AND SEARCH:")

    print(
        result_service.format_results(
            query1["id"]
        )
    )

    #OR QUERY

    query2 = query_service.create_query({

        "terms": ["python", "microservices"],

        "operator": "OR"
    })

    print("\nOR SEARCH:")

    print(
        result_service.format_results(
            query2["id"]
        )
    )


main()

AND SEARCH:
{'id': '1', 'query_id': '1', 'formatted_results': [{'doc_id': '1', 'title': 'Python Programming', 'snippet': 'Python is a popular programming language for cloud computing...', 'score': 2}, {'doc_id': '2', 'title': 'Cloud Services', 'snippet': 'Cloud computing enables scalable microservices architecture...', 'score': 2}], 'count': 2}

OR SEARCH:
{'id': '2', 'query_id': '2', 'formatted_results': [{'doc_id': '3', 'title': 'Microservices', 'snippet': 'Microservices architecture improves scalability...', 'score': 1}, {'doc_id': '1', 'title': 'Python Programming', 'snippet': 'Python is a popular programming language for cloud computing...', 'score': 1}, {'doc_id': '2', 'title': 'Cloud Services', 'snippet': 'Cloud computing enables scalable microservices architecture...', 'score': 1}], 'count': 3}


In [7]:
class IndexService:

    def __init__(self):
        self.documents = {}
        self.index = {}

    def add_document(self, doc_data):

        doc_id = str(len(self.documents) + 1)

        self.documents[doc_id] = {
            **doc_data,
            "id": doc_id
        }

        words = doc_data["content"].lower().split()

        for word in words:

            if word not in self.index:
                self.index[word] = set()

            self.index[word].add(doc_id)

        return self.documents[doc_id]

    def get_document(self, doc_id):

        return self.documents.get(doc_id)

    def search_word(self, word):

        return list(
            self.index.get(word.lower(), set())
        )


class QueryService:

    def __init__(self, index_service):

        self.index_service = index_service
        self.queries = {}

    def create_query(self, query_data):

        query_id = str(len(self.queries) + 1)

        search_terms = query_data["terms"]

        operator = query_data.get(
            "operator",
            "AND"
        ).upper()

        #AND SEARCH

        if operator == "AND":

            results = None

            for term in search_terms:

                doc_ids = set(
                    self.index_service.search_word(term)
                )

                if results is None:
                    results = doc_ids

                else:
                    results = results & doc_ids

            if results is None:
                results = set()

        #OR SEARCH

        elif operator == "OR":

            results = set()

            for term in search_terms:

                doc_ids = set(
                    self.index_service.search_word(term)
                )

                results = results | doc_ids

        else:

            return {
                "error": "Invalid operator"
            }

        #RANKING

        ranked_results = []

        for doc_id in results:

            doc = self.index_service.get_document(doc_id)

            score = 0

            for term in search_terms:

                score += doc["content"].lower().count(
                    term.lower()
                )

            ranked_results.append({

                "doc_id": doc_id,

                "score": score
            })

        #SORT RESULTS BY SCORE

        ranked_results.sort(

            key=lambda x: x["score"],

            reverse=True
        )

        query = {

            "id": query_id,

            "terms": search_terms,

            "operator": operator,

            "results": ranked_results
        }

        self.queries[query_id] = query

        return query


class ResultService:

    def __init__(
        self,
        index_service,
        query_service
    ):

        self.index_service = index_service

        self.query_service = query_service

        self.results = {}

    def format_results(self, query_id):

        query = self.query_service.queries.get(query_id)

        if not query:

            return {
                "error": "Query not found"
            }

        formatted_results = []

        for item in query["results"]:

            doc_id = item["doc_id"]

            score = item["score"]

            doc = self.index_service.get_document(doc_id)

            if doc:

                formatted_results.append({

                    "doc_id": doc_id,

                    "title": doc["title"],

                    "snippet":
                    doc["content"][:100] + "...",

                    "score": score
                })

        result_id = str(len(self.results) + 1)

        result = {

            "id": result_id,

            "query_id": query_id,

            "formatted_results": formatted_results,

            "count": len(formatted_results)
        }

        self.results[result_id] = result

        return result


#PRINT RESULTS

def print_results(title, result):

    print("\n" + "=" * 60)

    print(title)

    print("=" * 60)

    print(f"\nQuery ID: {result['query_id']}")

    print(f"Number of Results: {result['count']}")

    print("\nReturned Documents:\n")

    for doc in result["formatted_results"]:

        print(f"Document ID: {doc['doc_id']}")

        print(f"Title: {doc['title']}")

        print(f"Snippet: {doc['snippet']}")

        print(f"Score: {doc['score']}")

        print("-" * 60)


def main():

    #INITIALIZE SERVICES

    index_service = IndexService()

    query_service = QueryService(
        index_service
    )

    result_service = ResultService(
        index_service,
        query_service
    )

    #ADD DOCUMENTS

    #DOCUMENT 1
    index_service.add_document({

        "title": "Python Programming",

        "content":
        "Python is a popular programming language for cloud computing"
    })

    #DOCUMENT 2
    index_service.add_document({

        "title": "Cloud Services",

        "content":
        "Cloud computing enables scalable microservices architecture"
    })

    #DOCUMENT 3
    #We added this document
    index_service.add_document({

        "title": "Microservices",

        "content":
        "Microservices architecture improves scalability"
    })

    #AND QUERY

    query1 = query_service.create_query({

        "terms": ["cloud", "computing"],

        "operator": "AND"
    })

    and_results = result_service.format_results(
        query1["id"]
    )

    print_results(
        "AND SEARCH RESULTS",
        and_results
    )

    #OR QUERY

    query2 = query_service.create_query({

        "terms": ["python", "microservices"],

        "operator": "OR"
    })

    or_results = result_service.format_results(
        query2["id"]
    )

    print_results(
        "OR SEARCH RESULTS",
        or_results
    )

main()


AND SEARCH RESULTS

Query ID: 1
Number of Results: 2

Returned Documents:

Document ID: 1
Title: Python Programming
Snippet: Python is a popular programming language for cloud computing...
Score: 2
------------------------------------------------------------
Document ID: 2
Title: Cloud Services
Snippet: Cloud computing enables scalable microservices architecture...
Score: 2
------------------------------------------------------------

OR SEARCH RESULTS

Query ID: 2
Number of Results: 3

Returned Documents:

Document ID: 3
Title: Microservices
Snippet: Microservices architecture improves scalability...
Score: 1
------------------------------------------------------------
Document ID: 1
Title: Python Programming
Snippet: Python is a popular programming language for cloud computing...
Score: 1
------------------------------------------------------------
Document ID: 2
Title: Cloud Services
Snippet: Cloud computing enables scalable microservices architecture...
Score: 1
------------------